In [2]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 193.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 24.7 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from google.colab import drive

drive.mount('/content/drive')

PATH = "/content/drive/MyDrive/코인데이터/ALL_1m.parquet"

# 1) 데이터 로드
df = pd.read_parquet(PATH)

# 2025년만 사용
df = df.loc["2025-01-01":"2025-12-31"]

results = []

for col in df.columns:

    series = df[col].dropna()
    series = series[series > 0]

    if len(series) < 5000:
        continue

    logp = np.log(series)

    # 레벨 ADF (unit root 존재 여부 확인)
    adf_level = adfuller(logp.values, regression="c", maxlag=1, autolag=None)
    p_level = adf_level[1]

    # 1차 차분 ADF
    diff1 = logp.diff().dropna()
    adf_diff = adfuller(diff1.values, regression="c", maxlag=1, autolag=None)
    p_diff = adf_diff[1]

    is_I1 = (p_level > 0.05) and (p_diff < 0.05)

    results.append({
        "coin": col,
        "p_level": p_level,
        "p_diff1": p_diff,
        "is_I(1)": is_I1,
        "n_obs": len(logp)
    })

result_df = pd.DataFrame(results).sort_values("p_diff1")

pd.set_option("display.max_rows", None)
print(result_df.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_458/2119438736.py:29: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_level = adfuller(logp.values, regression="c", maxlag=1, autolag=None)
/tmp/ipykernel_458/2119438736.py:34: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_diff = adfuller(diff1.values, regression="c", maxlag=1, autolag=None)
/tmp/ipykernel_458/2119438736.py:29: FutureWarning: adfuller

    coin  p_level  p_diff1  is_I(1)  n_obs
 KRW-ADA 0.762006      0.0     True 509971
KRW-AVAX 0.553303      0.0     True 447091
 KRW-BTC 0.338192      0.0     True 524040
KRW-DOGE 0.301419      0.0     True 521117
 KRW-DOT 0.804882      0.0     True 385260
 KRW-ETH 0.653105      0.0     True 523972
KRW-LINK 0.319265      0.0     True 480235
 KRW-SOL 0.333848      0.0     True 523715
 KRW-TRX 0.011392      0.0    False 481647
 KRW-XRP 0.101004      0.0     True 524048


/tmp/ipykernel_458/2119438736.py:34: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_diff = adfuller(diff1.values, regression="c", maxlag=1, autolag=None)


In [5]:
# 2) VAR lag 선택 (AIC/BIC) + 바스켓 생성 + basket_key 생성

import numpy as np
import pandas as pd
import itertools
from statsmodels.tsa.vector_ar.var_model import VAR
from google.colab import drive

drive.mount('/content/drive')

PATH = "/content/drive/MyDrive/코인데이터/ALL_1m.parquet"

BASE = "KRW-BTC"
MAXLAGS = 5
MIN_ROWS = 20000

df = pd.read_parquet(PATH)
df = df.loc["2025-01-01":"2025-12-31"].sort_index()

# 1) 단계에서 I(1)으로 판정된 코인만 사용
I1_COINS = result_df.loc[result_df["is_I(1)"] == True, "coin"].tolist()
if BASE not in I1_COINS:
    raise ValueError(f"{BASE}가 I(1) 조건을 만족하지 않아 바스켓을 구성할 수 없습니다.")

all_coins = [c for c in I1_COINS if c in df.columns]
others = [c for c in all_coins if c != BASE]

baskets = []
for combo in itertools.combinations(others, 2):
    baskets.append([BASE] + list(combo))
for combo in itertools.combinations(others, 3):
    baskets.append([BASE] + list(combo))

def prep_logp_for_basket(basket):
    prices = df[basket].dropna(how="any")
    prices = prices[(prices > 0).all(axis=1)]
    if len(prices) < MIN_ROWS:
        return None
    return np.log(prices)

lag_rows = []

for B in baskets:
    logp = prep_logp_for_basket(B)
    if logp is None:
        continue
    try:
        sel = VAR(logp.values).select_order(maxlags=MAXLAGS)
        p_aic = sel.aic
        p_bic = sel.bic

        p_used = p_aic if p_aic is not None else p_bic
        if p_used is None:
            p_used = 1
        p_used = max(int(p_used), 1)

        k_ar_diff = max(p_used - 1, 0)

        lag_rows.append({
            "basket_key": "|".join(B),
            "basket": B,
            "n_rows": len(logp),
            "p_aic": None if p_aic is None else int(p_aic),
            "p_bic": None if p_bic is None else int(p_bic),
            "p_used": p_used,
            "k_ar_diff": k_ar_diff,
        })
    except Exception:
        continue

lag_df = pd.DataFrame(lag_rows).sort_values(["n_rows"], ascending=False)
print("lag_df rows:", len(lag_df))
print(lag_df.head(10).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
lag_df rows: 84
                      basket_key                                basket  n_rows  p_aic  p_bic  p_used  k_ar_diff
         KRW-BTC|KRW-ETH|KRW-XRP           [KRW-BTC, KRW-ETH, KRW-XRP]  523960      5      5       5          4
         KRW-BTC|KRW-SOL|KRW-XRP           [KRW-BTC, KRW-SOL, KRW-XRP]  523703      5      5       5          4
         KRW-BTC|KRW-ETH|KRW-SOL           [KRW-BTC, KRW-ETH, KRW-SOL]  523631      5      5       5          4
 KRW-BTC|KRW-ETH|KRW-SOL|KRW-XRP  [KRW-BTC, KRW-ETH, KRW-SOL, KRW-XRP]  523629      5      5       5          4
        KRW-BTC|KRW-DOGE|KRW-XRP          [KRW-BTC, KRW-DOGE, KRW-XRP]  521106      5      5       5          4
        KRW-BTC|KRW-DOGE|KRW-ETH          [KRW-BTC, KRW-DOGE, KRW-ETH]  521036      5      5       5          4
KRW-BTC|KRW-DOGE|KRW-ETH|KRW-XRP [KRW-BTC, KRW-DOGE, KRW-ETH, KRW-XRP] 

In [6]:
# 3) Johansen test (rank 결정) + candidates_df 생성

from statsmodels.tsa.vector_ar.vecm import coint_johansen

DET_ORDER = 0

def choose_rank_trace(jres, alpha=0.05):
    col = {0.10: 0, 0.05: 1, 0.01: 2}[alpha]
    rank = 0
    for r in range(len(jres.lr1)):
        if jres.lr1[r] > jres.cvt[r, col]:
            rank = r + 1
    return rank

joh_rows = []

for _, row in lag_df.iterrows():
    B = row["basket"]
    k_ar_diff = int(row["k_ar_diff"])
    basket_key = row["basket_key"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    try:
        jres = coint_johansen(logp.values, det_order=DET_ORDER, k_ar_diff=k_ar_diff)
        r10 = choose_rank_trace(jres, 0.10)
        r05 = choose_rank_trace(jres, 0.05)
        r01 = choose_rank_trace(jres, 0.01)

        joh_rows.append({
            "basket_key": basket_key,
            "basket": B,
            "k_ar_diff": k_ar_diff,
            "rank_10%": r10,
            "rank_5%": r05,
            "rank_1%": r01,
        })
    except Exception:
        continue

joh_df = pd.DataFrame(joh_rows).sort_values(["rank_5%", "rank_10%"], ascending=[False, False])
print("joh_df rows:", len(joh_df))
print(joh_df.head(20).to_string(index=False))

candidates_df = joh_df[joh_df["rank_5%"] >= 1].copy()
print("\ncandidates (rank_5%>=1):", len(candidates_df))

joh_df rows: 84
                        basket_key                                  basket  k_ar_diff  rank_10%  rank_5%  rank_1%
  KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP   [KRW-BTC, KRW-LINK, KRW-SOL, KRW-XRP]          4         4        1        1
 KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-XRP  [KRW-BTC, KRW-AVAX, KRW-DOGE, KRW-XRP]          4         4        1        0
          KRW-BTC|KRW-LINK|KRW-SOL            [KRW-BTC, KRW-LINK, KRW-SOL]          4         3        1        1
           KRW-BTC|KRW-ETH|KRW-SOL             [KRW-BTC, KRW-ETH, KRW-SOL]          4         1        1        0
  KRW-BTC|KRW-DOGE|KRW-ETH|KRW-SOL   [KRW-BTC, KRW-DOGE, KRW-ETH, KRW-SOL]          4         1        1        0
  KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL   [KRW-BTC, KRW-ETH, KRW-LINK, KRW-SOL]          4         1        1        1
          KRW-BTC|KRW-AVAX|KRW-ETH            [KRW-BTC, KRW-AVAX, KRW-ETH]          4         1        1        0
  KRW-BTC|KRW-AVAX|KRW-ETH|KRW-XRP   [KRW-BTC, KRW-AVAX, KRW-ETH, KRW-XR

In [7]:
# 4) 공적분 벡터 가중치(weights) 생성 (rank 개수만큼), weights_df 생성

def normalize_on_anchor(vec, anchor_idx):
    v = vec.astype(float).copy()
    if abs(v[anchor_idx]) < 1e-12:
        v = v / (np.max(np.abs(v)) + 1e-12)
    else:
        v = v / v[anchor_idx]
    return v

weight_rows = []

for _, row in candidates_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    k_ar_diff = int(row["k_ar_diff"])
    rank = int(row["rank_5%"])

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    cols = list(logp.columns)
    anchor_idx = cols.index(BASE)

    try:
        jres = coint_johansen(logp.values, det_order=DET_ORDER, k_ar_diff=k_ar_diff)
    except Exception:
        continue

    for i in range(rank):
        w = normalize_on_anchor(jres.evec[:, i], anchor_idx)
        weights = pd.Series(w, index=cols).to_dict()

        weight_rows.append({
            "basket_key": basket_key,
            "basket": B,
            "k_ar_diff": k_ar_diff,
            "rank_5%": rank,
            "vector_number": i + 1,
            "weights": weights
        })

weights_df = pd.DataFrame(weight_rows).sort_values(
    ["rank_5%", "basket_key", "vector_number"],
    ascending=[False, True, True]
)

print("weights_df rows:", len(weights_df))
print(weights_df.head(10).to_string(index=False))

weights_df rows: 12
                       basket_key                                 basket  k_ar_diff  rank_5%  vector_number                                                                                                               weights
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-ETH [KRW-BTC, KRW-AVAX, KRW-DOGE, KRW-ETH]          4        1              1    {'KRW-BTC': 1.0, 'KRW-AVAX': -0.6963666367941299, 'KRW-DOGE': 0.6384568951584157, 'KRW-ETH': -0.24606769402252981}
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-XRP [KRW-BTC, KRW-AVAX, KRW-DOGE, KRW-XRP]          4        1              1       {'KRW-BTC': 1.0, 'KRW-AVAX': -2.939706723767662, 'KRW-DOGE': 4.054854883115359, 'KRW-XRP': -2.4044025889215885}
 KRW-BTC|KRW-AVAX|KRW-DOT|KRW-ETH  [KRW-BTC, KRW-AVAX, KRW-DOT, KRW-ETH]          4        1              1     {'KRW-BTC': 1.0, 'KRW-AVAX': -0.3687120912890558, 'KRW-DOT': 0.2217259564540686, 'KRW-ETH': -0.09744042454866851}
         KRW-BTC|KRW-AVAX|KRW-ETH           [KRW-BTC, KRW-AVAX, KRW-ETH]    

In [8]:
# 5) 스프레드 정상성 확인 (ADF), spread_adf_df 생성 (안전 버전)

from statsmodels.tsa.stattools import adfuller

def build_spread(logp_df, weights_dict):
    w = np.array([weights_dict[c] for c in logp_df.columns], dtype=float)
    return pd.Series(logp_df.values @ w, index=logp_df.index)

spread_adf_rows = []

cnt_total = 0
cnt_logp_none = 0
cnt_spread_short = 0
cnt_adf_fail = 0
cnt_ok = 0

for _, row in weights_df.iterrows():
    cnt_total += 1

    B = row["basket"]
    basket_key = row["basket_key"]
    weights = row["weights"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        cnt_logp_none += 1
        continue

    spread = build_spread(logp, weights).dropna().astype(float)
    if len(spread) < 2000:
        cnt_spread_short += 1
        continue

    try:
        stat, pval, usedlag, nobs, crit = adfuller(
            spread.values,
            regression="c",
            maxlag=1,
            autolag=None
        )
        spread_adf_rows.append({
            "basket_key": basket_key,
            "vector_number": int(row["vector_number"]),
            "spread_adf_p": float(pval),
            "spread_adf_stat": float(stat),
            "nobs": int(nobs),
            "used_lag": int(usedlag),
        })
        cnt_ok += 1
    except Exception:
        cnt_adf_fail += 1
        continue

print("weights_df rows:", len(weights_df))
print("total tried:", cnt_total)
print("skipped (logp None):", cnt_logp_none)
print("skipped (spread too short):", cnt_spread_short)
print("failed (adfuller exception):", cnt_adf_fail)
print("ok:", cnt_ok)

if len(spread_adf_rows) == 0:
    print("spread_adf_rows is empty. No ADF results to sort.")
    spread_adf_df = pd.DataFrame(columns=[
        "basket_key","vector_number","spread_adf_p","spread_adf_stat","nobs","used_lag"
    ])
else:
    spread_adf_df = pd.DataFrame(spread_adf_rows).sort_values("spread_adf_p")

print("spread_adf_df rows:", len(spread_adf_df))
print(spread_adf_df.head(20).to_string(index=False))

/tmp/ipykernel_458/1557187765.py:35: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pval, usedlag, nobs, crit = adfuller(
/tmp/ipykernel_458/1557187765.py:35: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pval, usedlag, nobs, crit = adfuller(
/tmp/ipykernel_458/1557187765.py:35: FutureWarning: adfuller currently returns a plain tuple whose length depends on the

weights_df rows: 12
total tried: 12
skipped (logp None): 0
skipped (spread too short): 0
failed (adfuller exception): 0
ok: 12
spread_adf_df rows: 12
                       basket_key  vector_number  spread_adf_p  spread_adf_stat   nobs  used_lag
 KRW-BTC|KRW-DOGE|KRW-DOT|KRW-ETH              1  7.836250e-08        -6.144260 384167         1
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-XRP              1  1.849995e-07        -5.979814 445727         1
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP              1  1.946906e-06        -5.513640 480050         1
 KRW-BTC|KRW-ETH|KRW-LINK|KRW-SOL              1  2.646419e-05        -4.961254 480013         1
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-ETH              1  3.811080e-05        -4.880166 445691         1
         KRW-BTC|KRW-LINK|KRW-SOL              1  5.403932e-05        -4.801451 480051         1
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-XRP              1  6.030736e-04        -4.222655 447046         1
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-SOL              1  2.959147e-02        -3.0

/tmp/ipykernel_458/1557187765.py:35: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pval, usedlag, nobs, crit = adfuller(


위 코드 결과는 아래와 같이 해석될 수 있는데

O 공적분 벡터는 20개 생성됨 → Johansen은 통과

O logp는 정상적으로 만들어짐

O 스프레드 길이도 충분함

X 그런데 ADF가 20개 전부 예외 발생

그래서 이후 코드 실행하지 못했음

In [9]:
# 6) AR(1) 적합 가능 확인 (b<0, R^2, Ljung-Box), ar1_df 생성

from statsmodels.stats.diagnostic import acorr_ljungbox

ar1_rows = []

for _, row in weights_df.iterrows():
    B = row["basket"]
    basket_key = row["basket_key"]
    weights = row["weights"]

    logp = prep_logp_for_basket(B)
    if logp is None:
        continue

    spread = build_spread(logp, weights).dropna().astype(float)
    if len(spread) < 2000:
        continue

    s_lag = spread.shift(1).dropna()
    ds = spread.diff().dropna()
    ds = ds.loc[s_lag.index]

    X = np.column_stack([np.ones(len(s_lag)), s_lag.values])
    y = ds.values

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    a, b = float(beta[0]), float(beta[1])

    yhat = X @ beta
    resid = y - yhat

    sse = float(np.sum((y - yhat) ** 2))
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - (sse / sst) if sst > 0 else np.nan

    try:
        lb = acorr_ljungbox(resid, lags=[20], return_df=True)
        lb_p = float(lb["lb_pvalue"].iloc[0])
    except Exception:
        lb_p = np.nan

    ar1_rows.append({
        "basket_key": basket_key,
        "vector_number": int(row["vector_number"]),
        "a": a,
        "b": b,
        "b_negative": (b < 0),
        "r2": float(r2),
        "ljungbox_p_lag20": lb_p,
        "n": int(len(y)),
    })

ar1_df = pd.DataFrame(ar1_rows)
print("ar1_df rows:", len(ar1_df))
print(ar1_df.sort_values(["b_negative", "ljungbox_p_lag20"], ascending=[False, False]).head(20).to_string(index=False))

ar1_df rows: 12
                       basket_key  vector_number         a         b  b_negative       r2  ljungbox_p_lag20      n
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-ETH              1  0.002389 -0.000208        True 0.000106               0.0 445692
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-XRP              1 -0.002800 -0.000339        True 0.000172               0.0 445728
 KRW-BTC|KRW-AVAX|KRW-DOT|KRW-ETH              1  0.001081 -0.000070        True 0.000037               0.0 344054
         KRW-BTC|KRW-AVAX|KRW-ETH              1  0.000610 -0.000042        True 0.000023               0.0 447048
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-SOL              1  0.000681 -0.000046        True 0.000026               0.0 446905
 KRW-BTC|KRW-AVAX|KRW-ETH|KRW-XRP              1  0.001058 -0.000068        True 0.000045               0.0 447047
 KRW-BTC|KRW-DOGE|KRW-DOT|KRW-ETH              1  0.001749 -0.000417        True 0.000212               0.0 384168
 KRW-BTC|KRW-DOGE|KRW-ETH|KRW-SOL              1  0.000568 -0.00

In [10]:
# 7) 반감기 계산 + 최종 결합/정렬 + 저장

def half_life_from_b(b):
    if b is None or np.isnan(b):
        return np.nan
    if b >= 0:
        return np.inf
    return float(-np.log(2) / b)

final = (weights_df
         .merge(spread_adf_df, on=["basket_key", "vector_number"], how="left")
         .merge(ar1_df, on=["basket_key", "vector_number"], how="left"))

final["half_life_minutes"] = final["b"].apply(half_life_from_b)

final = final.sort_values(
    by=["rank_5%", "spread_adf_p", "half_life_minutes"],
    ascending=[False, True, True]
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print(final[[
    "basket_key", "basket", "rank_5%", "vector_number", "k_ar_diff",
    "spread_adf_p", "b", "ljungbox_p_lag20", "half_life_minutes", "weights"
]].to_string(index=False))

final.to_csv("cointegration_screen_1m_2025.csv", index=False, encoding="utf-8-sig")
print("\nSaved: cointegration_screen_1m_2025.csv")

                       basket_key                                 basket  rank_5%  vector_number  k_ar_diff  spread_adf_p         b  ljungbox_p_lag20  half_life_minutes                                                                                                               weights
 KRW-BTC|KRW-DOGE|KRW-DOT|KRW-ETH  [KRW-BTC, KRW-DOGE, KRW-DOT, KRW-ETH]        1              1          4  7.836250e-08 -0.000417               0.0        1661.664276      {'KRW-BTC': 1.0, 'KRW-DOGE': 1.2880428590697386, 'KRW-DOT': -1.0995562123395048, 'KRW-ETH': -0.8154702583156782}
KRW-BTC|KRW-AVAX|KRW-DOGE|KRW-XRP [KRW-BTC, KRW-AVAX, KRW-DOGE, KRW-XRP]        1              1          4  1.849995e-07 -0.000339               0.0        2046.721700       {'KRW-BTC': 1.0, 'KRW-AVAX': -2.939706723767662, 'KRW-DOGE': 4.054854883115359, 'KRW-XRP': -2.4044025889215885}
 KRW-BTC|KRW-LINK|KRW-SOL|KRW-XRP  [KRW-BTC, KRW-LINK, KRW-SOL, KRW-XRP]        1              1          4  1.946906e-06 -0.000202        